# openEO backup — Sentinel-2 indices and dNBR

This compact workflow replaces the main cloud-EO steps used in:

- **Practical 01:** Sentinel-2 orientation, NDVI and NBR;
- **Practical 03:** pre/post NBR and dNBR;
- **Practical 05:** pre-fire NDVI and NDMI.

The canonical GEE notebooks remain the primary course material. Use this notebook only when a trainer switches the class to the CDSE openEO backup.

To keep processing fast, the workflow uses a small buffered bounding box around EFFIS target polygon **240575**, not the full course AOI.

## 1. Imports and course target area

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import openeo
import rasterio

def find_repo_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parents[1] if len(Path.cwd().parents) > 1 else Path.cwd(),
        Path.home() / "mystorage" / "fire-school",
    ]
    for candidate in candidates:
        if (candidate / "data/effis/Galicica.gpkg").exists():
            return candidate
    raise FileNotFoundError("Could not find the fire-school repository.")

REPO_ROOT = find_repo_root()
EFFIS_PATH = REPO_ROOT / "data/effis/Galicica.gpkg"

effis = gpd.read_file(EFFIS_PATH).to_crs("EPSG:4326")
target = effis[effis["id"].astype(str) == "240575"].copy()
if target.empty:
    raise RuntimeError("EFFIS target polygon 240575 was not found.")

west, south, east, north = target.total_bounds
BUFFER_DEG = 0.025

BBOX = {
    "west": float(west - BUFFER_DEG),
    "south": float(south - BUFFER_DEG),
    "east": float(east + BUFFER_DEG),
    "north": float(north + BUFFER_DEG),
    "crs": "EPSG:4326",
}

print("Backup processing extent:", BBOX)

## 2. Connect and authenticate

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()
print("openEO authenticated.")

## 3. Reusable Sentinel-2 composite helper

CDSE provides the `to_scl_dilation_mask` process for Sentinel-2 Scene Classification Layer masking. This is slightly different from the simple SCL mask in the canonical GEE notebooks, so exact pixels may differ.

In [ ]:
MAX_CLOUD = 80

def s2_median(start, end, bands):
    scl = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=BBOX,
        temporal_extent=[start, end],
        bands=["SCL"],
        max_cloud_cover=MAX_CLOUD,
    )

    cloud_mask = scl.process(
        "to_scl_dilation_mask",
        data=scl,
        kernel1_size=17,
        kernel2_size=77,
        mask1_values=[2, 4, 5, 6, 7],
        mask2_values=[3, 8, 9, 10, 11],
        erosion_kernel_size=3,
    )

    data = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=BBOX,
        temporal_extent=[start, end],
        bands=bands,
        max_cloud_cover=MAX_CLOUD,
    )

    return data.mask(cloud_mask).reduce_temporal("median")

## 4. Build the pre/post fire composites and indices

In [ ]:
pre = s2_median(
    "2024-06-01",
    "2024-08-05",
    ["B04", "B08", "B11", "B12"],
)

post = s2_median(
    "2024-08-19",
    "2024-10-01",
    ["B04", "B08", "B11", "B12"],
)

prefire_july = s2_median(
    "2024-07-01",
    "2024-08-05",
    ["B04", "B08", "B11"],
)

def normalized_difference(cube, band_a, band_b):
    a = cube.band(band_a)
    b = cube.band(band_b)
    return (a - b) / (a + b)

nbr_pre_raw = normalized_difference(pre, "B08", "B12")
nbr_post_raw = normalized_difference(post, "B08", "B12")
dnbr_raw = nbr_pre_raw - nbr_post_raw

ndvi_prefire_raw = normalized_difference(prefire_july, "B08", "B04")
ndmi_prefire_raw = normalized_difference(prefire_july, "B08", "B11")

nbr_pre = nbr_pre_raw.rename_labels("bands", ["NBR_pre"])
nbr_post = nbr_post_raw.rename_labels("bands", ["NBR_post"])
dnbr = dnbr_raw.rename_labels("bands", ["dNBR"])
ndvi_prefire = ndvi_prefire_raw.rename_labels("bands", ["NDVI_prefire"])
ndmi_prefire = ndmi_prefire_raw.rename_labels("bands", ["NDMI_prefire"])

bundle = (
    nbr_pre
    .merge_cubes(nbr_post)
    .merge_cubes(dnbr)
    .merge_cubes(ndvi_prefire)
    .merge_cubes(ndmi_prefire)
)

print("Process graph ready. No cloud processing has run yet.")

## 5. Execute the compact target-area request

For this small backup extent, a synchronous download is attempted first. If it times out, the cell automatically switches to a batch job.

In [ ]:
OUTPUT_DIR = Path("/tmp/geo_adapt_openeo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = OUTPUT_DIR / "s2_indices_dnbr.tif"

try:
    bundle.download(str(OUTPUT))
    print("Synchronous result:", OUTPUT)
except Exception as exc:
    print("Synchronous request did not finish:", exc)
    print("Switching to a batch job...")
    bundle.execute_batch(
        outputfile=str(OUTPUT),
        title="GEO-ADAPT openEO backup: S2 indices + dNBR",
    )
    print("Batch result:", OUTPUT)

## 6. Inspect the result

In [ ]:
BAND_NAMES = [
    "NBR pre-fire",
    "NBR post-fire",
    "dNBR",
    "NDVI pre-fire",
    "NDMI pre-fire",
]

with rasterio.open(OUTPUT) as src:
    data = src.read().astype("float32")
    nodata = src.nodata

if nodata is not None:
    data[data == nodata] = np.nan

print("Bands:", data.shape[0])
print("Raster size:", data.shape[2], "×", data.shape[1])

for i, name in enumerate(BAND_NAMES):
    arr = data[i]
    print(
        f"{name:16s}",
        "min =", round(float(np.nanmin(arr)), 3),
        "median =", round(float(np.nanmedian(arr)), 3),
        "max =", round(float(np.nanmax(arr)), 3),
    )

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(arr)
    ax.set_title(name)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)
    plt.show()

## 7. Interpretation questions

1. Is the 2024 disturbance visible as higher dNBR in a spatially coherent area?
2. Do the pre-fire NDVI and NDMI maps emphasize the same places?
3. Why might this result differ slightly from the GEE notebook even with the same dates?
4. Which differences are caused by backend implementation, and which would change the scientific conclusion?

The backend is secondary. The evidence and its limitations remain the focus.